In [1]:
#import settings

import pandas as pd
import lightgbm as lgb 
from sklearn.preprocessing import MultiLabelBinarizer

from recommenders.utils.timer import Timer
from recommenders.datasets import movielens
from recommenders.datasets.python_splitters import python_stratified_split
from recommenders.evaluation.python_evaluation import (
    rmse,
    mae,
    rsquared,
    exp_var
)

c:\Users\muzic\AppData\Local\Programs\Python\Python310\lib\site-packages\pandera\_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [2]:
#parameter settings

#top k items to recommend
TOP_K = 10

#select movielens data size:
MOVIELENS_DATA_SIZE = '100k' #options: '100k', '1m', '10m', '20m'

#other data settings
USER_COL = "userID"
ITEM_COL = "itemID"
RATING_COL = "rating"
PREDICTION_COL = "prediction"
ITEM_FEAT_COL = "genres"

#Train test split ratio
SPLIT_RATIO = 0.75

#Model Settings
MAX_LEAF = 64
NUM_OF_TREES = 100
LEARNING_RATE = 0.05
METRIC = "mae"

SEED = 42 

In [3]:
params = {
    "objective":"regression",
    "boosting_type":"gbdt",
    "metric":METRIC,
    "num_leaves":MAX_LEAF,
    "boost_from_average":True,
    "n_jobs": -1,
    "learning_rate":LEARNING_RATE,
}

In [4]:
#data preparation

data = movielens.load_pandas_df(
    size = MOVIELENS_DATA_SIZE,
    header = [USER_COL, ITEM_COL, RATING_COL],
    genres_col = ITEM_FEAT_COL
)
data.head()

100%|██████████| 4.81k/4.81k [00:00<00:00, 9.81kKB/s]


,userID,itemID,rating,genres
0,196,242,3.0,Comedy
1,186,302,3.0,Crime|Film-Noir|Mystery|Thriller
2,22,377,1.0,Children's|Comedy
3,244,51,2.0,Drama|Romance|War|Western
4,166,346,1.0,Crime|Drama


In [5]:
#encode item features with multi-hot encoding with MultiLabelBinarizer

genres_encoder = MultiLabelBinarizer()
data[ITEM_FEAT_COL] = genres_encoder.fit_transform(
    data[ITEM_FEAT_COL].apply(lambda s: s.split("|"))
).tolist()

print("Genres: ", genres_encoder.classes_)
data.head()

Genres:  ['Action' 'Adventure' 'Animation' "Children's" 'Comedy' 'Crime'
 'Documentary' 'Drama' 'Fantasy' 'Film-Noir' 'Horror' 'Musical' 'Mystery'
 'Romance' 'Sci-Fi' 'Thriller' 'War' 'Western' 'unknown']


,userID,itemID,rating,genres
0,196,242,3.0,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,186,302,3.0,"[0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, ..."
2,22,377,1.0,"[0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,244,51,2.0,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ..."
4,166,346,1.0,"[0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."


In [6]:
#Expand the 'genre' list into separate columns 
number_of_genres = len(genres_encoder.classes_)
expanded_genre = pd.DataFrame(data[ITEM_FEAT_COL].tolist(), columns=[f"{ITEM_FEAT_COL}_{i+1}" for i in range(number_of_genres)])

#Concatenate the expanded genre columns with the original DataFrame
data = pd.concat([data,expanded_genre], axis=1)

#Drop the original 'genre' column
data.drop(ITEM_FEAT_COL, axis=1, inplace =True)
data.head()

,userID,itemID,rating,genres_1,genres_2,genres_3,genres_4,genres_5,genres_6,genres_7,genres_8,genres_9,genres_10,genres_11,genres_12,genres_13,genres_14,genres_15,genres_16,genres_17,genres_18,genres_19
0,196,242,3.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,186,302,3.0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0,0,0
2,22,377,1.0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,244,51,2.0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,1,0
4,166,346,1.0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0


In [7]:
#Split the data into train and test sets with python_stratified_split

train,test = python_stratified_split(data,ratio = SPLIT_RATIO, col_user = USER_COL, col_item=ITEM_COL, seed=SEED)

In [8]:
print("""
    Train:
    Total Ratings : {train_total}
    Unique Users : {train_users}
    Unique Items : {train_items}
      
    Test:
    Total Ratings : {test_total}
    Unique Users : {test_users}
    Unique Items : {test_items}
    """.format(
        train_total = len(train),
        train_users = len(train[USER_COL].unique()),
        train_items = len(train[ITEM_COL].unique()),
        test_total = len(test),
        test_users = len(test[USER_COL].unique()),
        test_items = len(test[ITEM_COL].unique())
    )
)


    Train:
    Total Ratings : 74992
    Unique Users : 943
    Unique Items : 1653
      
    Test:
    Total Ratings : 25008
    Unique Users : 943
    Unique Items : 1444
    


In [9]:
#Train the LightGBM Model

lgb_regressor = lgb.LGBMRegressor(**params)

In [10]:
with Timer() as train_time:
    lgb_regressor.fit(
        X = train[train.columns.difference([RATING_COL])].values,
        y = train[RATING_COL].values,
    )
print(f"Took {train_time.interval} seconds for training.")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008279 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 543
[LightGBM] [Info] Number of data points in the train set: 74992, number of used features: 20
[LightGBM] [Info] Start training from score 3.529670
Took 0.6016168000060134 seconds for training.


In [11]:
#evaluate the model

with Timer() as test_time:
    y_pred = lgb_regressor.predict(test[test.columns.difference([RATING_COL])])

print(f"Took {test_time.interval} seconds for prediction.")

Took 0.06541250000009313 seconds for prediction.


In [12]:
pred = test[[USER_COL, ITEM_COL, RATING_COL]].copy()
pred[PREDICTION_COL] = y_pred
pred.head()

,userID,itemID,rating,prediction
56881,1,25,4.0,3.438477
81463,1,126,2.0,4.014069
56067,1,45,5.0,3.926278
14425,1,255,2.0,3.434972
35389,1,144,4.0,3.685498


In [13]:
#Rating metrics
eval_rmse = rmse(test, pred, col_user=USER_COL, col_item=ITEM_COL,col_rating=RATING_COL, col_prediction= PREDICTION_COL)
eval_mae = mae(test, pred, col_user=USER_COL, col_item=ITEM_COL,col_rating=RATING_COL, col_prediction= PREDICTION_COL)
eval_rsquared = rsquared(test, pred, col_user=USER_COL, col_item=ITEM_COL,col_rating=RATING_COL, col_prediction= PREDICTION_COL)
eval_exp_var = exp_var(test, pred, col_user=USER_COL, col_item=ITEM_COL,col_rating=RATING_COL, col_prediction= PREDICTION_COL)

In [14]:
print("Model:\t\tLightGBM",
      "RMSE:\t\t%f" %eval_rmse,
      "MAE:\t\t%f" %eval_mae,
      "R2:\t\t%f" %eval_rsquared,
      "Explained Variance:\t%f" %eval_exp_var,
      sep ='\n')


Model:		LightGBM
RMSE:		1.006508
MAE:		0.806876
R2:		0.201780
Explained Variance:	0.201782
